### Imports

In [1]:
! apt install python3.10-venv -y -q
!python3 -m venv venv
!source venv/bin/activate

Reading package lists...
Building dependency tree...
Reading state information...
E: Unable to locate package python3.10-venv
E: Couldn't find any package by glob 'python3.10-venv'


In [2]:
import os
import json
import base64
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
from skimage import io
from skimage.color import rgb2gray
from skimage.measure import label, regionprops
from skimage.util import img_as_ubyte
from numpy import pad
import requests
from PIL import Image
from numpy import asarray
import cv2
from skimage.transform import resize
from skimage.util import view_as_blocks
from skimage import io, transform, util, img_as_ubyte
from skimage.transform import resize
from skimage.util import view_as_blocks
from skimage.color import rgb2gray
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
import pandas as pd
import zipfile
from scipy import spatial
import skimage.util

### Datasets

In [3]:
images_dir = '/kaggle/input/ihc-cd8-img-2/Auploader/'
output_dir = '/kaggle/working/'
filename = '4_Lenti-HPV-07_CD8.tif'

In [4]:
res = requests.post(
    url='https://deepliif.org/api/infer',
    files={
        'img': open(f'{images_dir}/{filename}', 'rb')
    },
    # optional param that can be 10x, 20x (default) or 40x
    params={
        'resolution': '20x'
    }
)

data = res.json()

def b64_to_pil(b):
    return Image.open(BytesIO(base64.b64decode(b.encode())))

for name, img in data['images'].items():
    output_filepath = f'{output_dir}/{os.path.splitext(filename)[0]}_{name}.png'
    with open(output_filepath, 'wb') as f:
        b64_to_pil(img).save(f, format='PNG')

print(json.dumps(data['scoring'], indent=2))

{
  "num_neg": 2147,
  "num_pos": 1181,
  "num_total": 3328,
  "percent_pos": 35.5,
  "prob_thresh": 80,
  "size_thresh": 50
}


In [5]:
!ls -F

4_Lenti-HPV-07_CD8_DAPI.png    4_Lenti-HPV-07_CD8_Seg.png
4_Lenti-HPV-07_CD8_Hema.png    4_Lenti-HPV-07_CD8_SegOverlaid.png
4_Lenti-HPV-07_CD8_Lap2.png    4_Lenti-HPV-07_CD8_SegRefined.png
4_Lenti-HPV-07_CD8_Marker.png  venv/


### Processing

In [6]:
def process_and_save_density_images(input_path, output_dir, colors, target_size=(100, 100)):
    # Load image
    image = io.imread(input_path)

    # Resize image
    resized_image = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)

    # Convert to grayscale
    resized_image_gray = rgb2gray(resized_image)

    # Initialize labeled masks
    labeled_tumor = np.zeros_like(resized_image_gray, dtype=bool)
    labeled_immune = np.zeros_like(resized_image_gray, dtype=bool)
    labeled_normal = np.zeros_like(resized_image_gray, dtype=bool)

    # Extract regions based on color ranges
    for channel in range(3):
        labeled_tumor = np.logical_or(
            labeled_tumor, np.logical_and(
                resized_image[..., channel] >= colors['tumor'][0][channel],
                resized_image[..., channel] <= colors['tumor'][1][channel]
            )
        )
        labeled_immune = np.logical_or(
            labeled_immune, np.logical_and(
                resized_image[..., channel] >= colors['immune'][0][channel],
                resized_image[..., channel] <= colors['immune'][1][channel]
            )
        )
        labeled_normal = np.logical_or(
            labeled_normal, np.logical_and(
                resized_image[..., channel] >= colors['normal'][0][channel],
                resized_image[..., channel] <= colors['normal'][1][channel]
            )
        )

    # Convert to density images
    tumor_density = img_as_ubyte(labeled_tumor)
    immune_density = img_as_ubyte(labeled_immune)
    normal_density = img_as_ubyte(labeled_normal)

    # Threshold density images
    tumor_density = tumor_density // 255
    immune_density = immune_density // 255
    normal_density = normal_density // 255

    # Save density images
    io.imsave(f'{output_dir}/resized_immune.png', immune_density)
    io.imsave(f'{output_dir}/resized_tumor.png', tumor_density)
    io.imsave(f'{output_dir}/resized_normal.png', normal_density)

In [7]:
colors = {'immune': [[200, 0, 0], [255, 13, 13]],
          'tumor': [[0, 0, 225], [13, 13, 255]],
          'normal': [[120, 120, 120], [135, 135, 135]]}

input_image_path = '/kaggle/working/4_Lenti-HPV-07_CD8_SegOverlaid.png'
output_directory = '/kaggle/working/'

process_and_save_density_images(input_image_path, output_directory, colors)

/tmp/ipykernel_47/484725117.py:48: UserWarning: /kaggle/working//resized_immune.png is a low contrast image
  io.imsave(f'{output_dir}/resized_immune.png', immune_density)
/tmp/ipykernel_47/484725117.py:49: UserWarning: /kaggle/working//resized_tumor.png is a low contrast image
  io.imsave(f'{output_dir}/resized_tumor.png', tumor_density)
/tmp/ipykernel_47/484725117.py:50: UserWarning: /kaggle/working//resized_normal.png is a low contrast image
  io.imsave(f'{output_dir}/resized_normal.png', normal_density)


### For Tumor cells

In [8]:
def process_and_save_tumor_data(input_path, output_dir, block_shape=(100, 100)):
    # Load tumor image
    tumor_img = io.imread(input_path)

    # View image as blocks and reshape
    tumor_img_blocks = view_as_blocks(tumor_img, block_shape=block_shape)
    tumor_img_flat = tumor_img_blocks.reshape(-1, *block_shape).ravel()

    # Save tumor density as CSV
    tumor_density_path = f'{output_dir}/Tum_dense.csv'
    np.savetxt(tumor_density_path, tumor_img_flat, delimiter=',', fmt='%d', header='Tum_dense', comments='')

    # Read CSV and create proliferating cells condition
    tumor_data = pd.read_csv(tumor_density_path)
    proliferating_cells_condition = (tumor_data["Tum_dense"] > 0)

    # Generate random values for proliferating cells
    tumor_img_prolif = np.zeros_like(tumor_data["Tum_dense"])
    tumor_img_prolif[proliferating_cells_condition] = np.random.randint(0, 10, np.sum(proliferating_cells_condition))

    # Threshold proliferating cells
    tumor_img_prolif = np.where(tumor_img_prolif > 0, 1, 0)

    # Save proliferating cells as CSV
    tumor_prolif_path = f'{output_dir}/Tum_prolif.csv'
    np.savetxt(tumor_prolif_path, tumor_img_prolif, delimiter=',', fmt='%d', header='Tum_prolif', comments='')

    # Reshape and save tumor IDs as CSV
    tumor_img_blocks_df = pd.DataFrame(tumor_img_blocks.ravel())
    tumor_img_blocks_df = tumor_img_blocks_df[tumor_img_blocks_df[0] >= 0]
    tumor_img_blocks_df = tumor_img_blocks_df.reset_index(drop=True)
    tumor_img_blocks_df = tumor_img_blocks_df.reset_index()
    tumor_img_blocks_df = tumor_img_blocks_df.rename(columns={'index': 'ID'})
    tumor_id_path = f'{output_dir}/tumor_id.csv'
    tumor_img_blocks_df.to_csv(tumor_id_path, index=False, header='id')

In [9]:
input_tumor_path = '/kaggle/working/resized_tumor.png'
output_directory_tumor = '/kaggle/working/'

process_and_save_tumor_data(input_tumor_path, output_directory_tumor)

### For Immune (CD8) cells

In [10]:
def process_and_save_immune_data(input_path, output_dir, block_shape=(100, 100)):
    # Load immune image
    immune_img = io.imread(input_path)

    # View image as blocks
    immune_img_blocks = view_as_blocks(immune_img, block_shape=block_shape)

    # Reshape and save immune IDs as CSV
    immune_img_blocks_df = pd.DataFrame(immune_img_blocks.ravel())
    immune_img_blocks_df = immune_img_blocks_df[immune_img_blocks_df[0] >= 0]
    immune_img_blocks_df = immune_img_blocks_df.reset_index(drop=True)
    immune_img_blocks_df = immune_img_blocks_df.reset_index()
    immune_img_blocks_df = immune_img_blocks_df.rename(columns={'index': 'ID'})
    immune_id_path = f'{output_dir}/immune_id.csv'
    immune_img_blocks_df.to_csv(immune_id_path, index=False, header='id')

    # Flatten and save CD8 dense as CSV
    immune_img_dense = immune_img_blocks.ravel()
    immune_img_dense = immune_img_dense.astype(np.uint8)
    cd8_dense_path = f'{output_dir}/CD8_dense.csv'
    np.savetxt(cd8_dense_path, immune_img_dense, delimiter=',', fmt='%d', header='CD8_dense', comments='')

    # Read CD8 dense CSV and create proliferating cells condition
    immune_data = pd.read_csv(cd8_dense_path)
    proliferating_cells_condition = (immune_data["CD8_dense"] > 0)

    # Generate random values for proliferating cells and threshold
    immune_img_prolif = np.zeros_like(immune_data["CD8_dense"])
    immune_img_prolif[proliferating_cells_condition] = np.random.randint(0, 8, np.sum(proliferating_cells_condition))
    immune_img_prolif = np.where(immune_img_prolif > 0, 1, 0)

    # Save proliferating cells as CSV
    cd8_prolif_path = f'{output_dir}/CD8_prolif.csv'
    np.savetxt(cd8_prolif_path, immune_img_prolif, delimiter=',', fmt='%d', header='CD8_prolif', comments='')

    # Read CD8 proliferating cells CSV and create non-proliferating cells condition
    immune_data_2 = pd.read_csv(cd8_prolif_path)
    non_prolif_condition = (immune_data_2["CD8_prolif"] == 0) & proliferating_cells_condition

    # Generate binary mask for non-proliferating cells
    immune_img_non_prolif = np.where(non_prolif_condition, 1, 0)

    # Save non-proliferating cells as CSV
    cd8_non_prolif_path = f'{output_dir}/CD8_non_prolif.csv'
    np.savetxt(cd8_non_prolif_path, immune_img_non_prolif, delimiter=',', fmt='%d', header='CD8_non_prolif', comments='')

In [11]:
input_immune_path = '/kaggle/working/resized_immune.png'
output_directory_immune = '/kaggle/working/'

process_and_save_immune_data(input_immune_path, output_directory_immune)

### Compute RDF

In [12]:
import torch
from torch.nn.functional import pairwise_distance
import torch.nn.functional as F

In [141]:
# def radial_distribution_function(image, radii, tile_size=20):
#     # Image to tiles 
#     print(image.shape)
#     tiles = image.unfold(-2, tile_size, tile_size).unfold(-1, tile_size, tile_size)
#     print(tiles.shape)
#     # Filter valid tiles
#     counts = image.flatten(1).sum(-1)
#     valid = (counts > 10) & (counts < 10000) 
#     tiles = image.unfold(-2, tile_size, tile_size).unfold(-1, tile_size, tile_size)
#     tiles = tiles[valid[:, None]]
#     # Compute pairwise distances between pixels within each tile
#     inds = torch.where(tiles > 0)
#     inds = inds[0][:, None, None] 
#     coords = torch.stack([inds, inds], dim=-1)[:, :, None, :]
#     dist = torch.cdist(coords, coords) 
#     # Histogram distances and normalize  
#     num_bins = len(radii) - 1
#     hist = torch.histc(dist, bins=num_bins, min=0, max=int(radii[-1]))
#     print(hist.shape)
#     areas = 3.14159 * ((radii[1:]**2) - (radii[:-1]**2)) 
#     print(tiles.shape)
#     tile_counts = tiles.squeeze(-3).sum((-2, -1))
#     print(tile_counts.shape)
#     tile_counts = tiles.flatten(0, 3).sum(1)[:, None].expand(-1, len(areas))
#     print(tile_counts.shape)
#     tile_counts = tile_counts.unsqueeze(1).unsqueeze(-1).expand(-1, -1, len(areas), -1)
#     print(tile_counts.shape)
#     tile_counts = tile_counts.flatten(1,3)  
#     print(tile_counts.shape)
#     areas_repeated = areas.repeat(len(tile_counts), 1)  
#     print(len(areas))
#     expected = areas_repeated * tile_counts * (tile_counts - 1)
#     print(expected.shape)
#     expected = torch.clamp(expected, min=1e-7) 
#     rdf = hist / expected
#     # Average 
#     return rdf.mean(0)

In [143]:
def radial_distribution_function(image, radii, tile_size=100, min_counts=10, max_counts=10000):
    # Extract tiles 
    tiles = image.unfold(-2, tile_size, tile_size).unfold(-1, tile_size, tile_size)
    # Get valid tiles
    counts = tiles.sum((-3, -2, -1))
    valid = (counts > min_counts) & (counts < max_counts)
    # Expand valid 
    valid_exp = valid.unsqueeze(2)
    # Index tiles
    tiles = tiles[valid_exp.expand(tiles.shape)]
    # Get coords of all CD8 T-cells
    nonzero_coords = torch.nonzero(tiles > 0, as_tuple=True)
    cd8_coords = torch.stack(nonzero_coords, dim=-1).float()
    # Compute pairwise distances
    dist = torch.cdist(cd8_coords, cd8_coords)  
    # Bin counts hist
    hist = torch.histc(dist, bins=len(radii)-1, min=0, max=radii[-1].item()).float()  
    # Normalize   
    areas = 3.14159 * ((radii[1:]**2) - (radii[:-1]**2))
    reshaped_tiles = tiles.view(tiles.shape[0], -1)
    tile_counts = reshaped_tiles.sum(dim=-2).float()[:, None]
    expected =  areas[:, None] * tile_counts * (tile_counts - 1)
    expected = torch.clamp(expected, min=1e-7)
    rdf = hist / expected
    # Average        
    return rdf.mean(0)

In [18]:
immune_density = io.imread('/kaggle/working/resized_immune.png')
immune_density_tensor = torch.tensor(immune_density, dtype=torch.float32, requires_grad = True).unsqueeze(0).unsqueeze(0)

# Define radii
radii = torch.tensor(torch.arange(0, 10, dtype=torch.float32), requires_grad=True)

/tmp/ipykernel_47/2632907115.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  radii = torch.tensor(torch.arange(0, 10, dtype=torch.float32), requires_grad=True)


In [144]:
rdf_out = radial_distribution_function(immune_density_tensor, radii)

In [145]:
rdf_out

tensor([6.3081e-05, 1.2348e-05, 2.5874e-05, 8.5299e-06, 2.3112e-05, 1.3120e-05,
        1.8806e-05, 7.3114e-06, 2.8758e-05], grad_fn=<MeanBackward1>)

In [136]:
loss = rdf_out.mean()

In [137]:
loss.backward()

In [89]:
loss.requires_grad

True

In [90]:
immune_density_tensor.requires_grad

True

In [91]:
radii.requires_grad

True

### Compute SAM

In [146]:
def computeSAM(rdf_sim, rdf_obs, obs_range_tol=0.2, frac_within_tol=0.7):
    num_dists = rdf_obs.shape[0]
    # Calculate acceptable range
    rdf_range = (rdf_obs.max() - rdf_obs.min())
    max_obs = torch.max(rdf_obs + obs_range_tol * rdf_range, torch.zeros_like(rdf_obs))
    min_obs = torch.max(torch.min(rdf_obs - obs_range_tol * rdf_range, torch.zeros_like(rdf_obs)), torch.zeros_like(rdf_obs))
    # Check if simulated RDF is within tolerance range
    within_tol = (rdf_sim > min_obs) & (rdf_sim < max_obs)
    # Calculate fraction of dists where sim RDF is within tol 
    frac_within = torch.sum(within_tol) / num_dists
    # SAM is fraction of dists above threshold
    sam = (frac_within >= frac_within_tol).float()
    sam = torch.autograd.Variable(sam, requires_grad=True)
    return sam, frac_within

In [147]:
# Sample inputs 
rdf_obs = torch.tensor(torch.randn(100), requires_grad=True)  
rdf_sim = torch.tensor(torch.randn(100), requires_grad=True)
sam, frac_within = computeSAM(rdf_sim, rdf_obs, obs_range_tol=0.2, frac_within_tol=0.5)
print(f"SAM: {sam}") 
print(f"Fraction Within Tolerance: {frac_within:.3f}")

SAM: 0.0
Fraction Within Tolerance: 0.290


/tmp/ipykernel_47/2990771087.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  rdf_obs = torch.tensor(torch.randn(100), requires_grad=True)
/tmp/ipykernel_47/2990771087.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  rdf_sim = torch.tensor(torch.randn(100), requires_grad=True)


In [156]:
sam.requires_grad

True

In [157]:
loss = sam.sum()  
loss.backward() 

In [158]:
rdf_sim.requires_grad

True

In [159]:
rdf_obs.requires_grad

True

In [153]:
rdf_out_2 = torch.tensor(torch.randn(9), requires_grad=True) 
sam, frac_within = computeSAM(rdf_out, rdf_out_2, obs_range_tol=0.2, frac_within_tol=0.6)
print(f"SAM: {sam}") 
print(f"Fraction Within Tolerance: {frac_within:.3f}")

SAM: 1.0
Fraction Within Tolerance: 0.778


/tmp/ipykernel_47/3677170294.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  rdf_out_2 = torch.tensor(torch.randn(9), requires_grad=True)


### Compute VarSAM

In [192]:
def computeVarSAM(rdf_sim, rdf_obs):
    # Extract first 15 RDF distances 
    rdf_sim_near = rdf_sim[:15]  
    rdf_obs_near = rdf_obs[:15]
    # Calculate ranges
    range_sim = rdf_sim_near.max() - rdf_sim_near.min()
    range_obs = rdf_obs_near.max() - rdf_obs_near.min()
    # VarSAM is ratio of ranges  
    # Clamp between 0 and 1
    var_sam = torch.min(range_obs / range_sim, torch.tensor(1.0)) 
    var_sam = torch.max(var_sam, torch.tensor(0.0))
    return var_sam

In [193]:
rdf_sim = torch.tensor(torch.randn(9), requires_grad=True)
rdf_obs = torch.tensor(torch.randn(9), requires_grad=True)

var_sam = computeVarSAM(rdf_sim, rdf_obs)
print(var_sam)

tensor(0.6908, grad_fn=<MaximumBackward0>)


/tmp/ipykernel_47/2978990393.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  rdf_sim = torch.tensor(torch.randn(9), requires_grad=True)
/tmp/ipykernel_47/2978990393.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  rdf_obs = torch.tensor(torch.randn(9), requires_grad=True)


In [162]:
loss = var_sam.sum()

In [163]:
loss.backward()

In [164]:
loss.requires_grad

True

In [165]:
rdf_sim.requires_grad

True

In [166]:
rdf_obs.requires_grad

True

In [202]:
rdf_out_2 = torch.tensor(torch.randn(9), requires_grad=True) 

/tmp/ipykernel_47/2972251169.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  rdf_out_2 = torch.tensor(torch.randn(9), requires_grad=True)


In [203]:
var_sam = computeVarSAM(rdf_out, rdf_out_2)
print(var_sam)

tensor(1., grad_fn=<MaximumBackward0>)


In [199]:
var_sam = computeVarSAM(rdf_sim, rdf_obs)
print(var_sam)

tensor(0.6908, grad_fn=<MaximumBackward0>)
